### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Llama_32_3B_Instruct_lora_fp16_r256_s25000_i3000_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.95,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            #clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, base_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 05-08 22:57:42 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 05-08 22:57:43 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 05-08 22:57:48 [config.py:585] This model supports multiple tasks: {'generate', 'reward', 'classify', 'embed', 'score'}. Defaulting to 'generate'.
INFO 05-08 22:57:48 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-08 22:57:49 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s25000_i3000_msl2048', speculative_config=None, tokenizer='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s25000_i3000_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-08 22:57:56 [loader.py:447] Loading weights took 5.14 seconds
INFO 05-08 22:57:56 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 5.356240 seconds
INFO 05-08 22:58:02 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/cd2e9bb533/rank_0_0 for vLLM's torch.compile
INFO 05-08 22:58:02 [backends.py:425] Dynamo bytecode transform time: 6.49 s


[rank0]:W0508 22:58:03.719000 11052 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 05-08 22:58:05 [backends.py:132] Cache the graph of shape None for later use
INFO 05-08 22:58:21 [backends.py:144] Compiling a graph for general shape takes 18.39 s
INFO 05-08 22:58:32 [monitor.py:33] torch.compile takes 24.87 s in total
INFO 05-08 22:58:32 [kv_cache_utils.py:566] GPU KV cache size: 33,840 tokens
INFO 05-08 22:58:32 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 33.05x
INFO 05-08 22:58:50 [gpu_model_runner.py:1534] Graph capturing finished in 18 secs, took 0.57 GiB
INFO 05-08 22:58:50 [core.py:151] init engine (profile, create kv cache, warmup model) took 54.17 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
import sys
sys.path.append(r'/home/vino/ML_Projects/Drawing_with_LLMs/drawing-with-llms')
import pandas as pd

df1=pd.read_csv(r'./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv',header=[0])
df2=pd.read_csv(r'./drawing-with-llms/test_filtered_2_batch_vqa_gemini_2o_kaggle.csv',header=[0])
#df3=pd.read_csv(r'./drawing-with-llms/gemini_25_pro_validation/train_filtered_1_batch_gpt4.csv',header=[0])
#print(df3.shape)
#df2=df2.drop_duplicates(['description'])
#df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df1.drop_duplicates(['description'])

print(df.shape)
df.head(2)

(71, 7)


,description,clean_svg,sl_score,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [7]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [8]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 15
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/5 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:03<00:44,  3.14s/it, est. speed input: 19.09
cessed prompts:  13%|▏| 2/15 [00:04<00:24,  1.88s/it, est. speed input: 29.18
cessed prompts:  20%|▏| 3/15 [00:04<00:12,  1.07s/it, est. speed input: 42.77
cessed prompts:  27%|▎| 4/15 [00:04<00:07,  1.44it/s, est. speed input: 55.70
cessed prompts:  33%|▎| 5/15 [00:04<00:04,  2.08it/s, est. speed input: 67.77
cessed prompts:  40%|▍| 6/15 [00:05<00:05,  1.79it/s, est. speed input: 70.34
cessed prompts:  47%|▍| 7/15 [00:05<00:03,  2.33it/s, est. speed input: 79.60
cessed prompts:  53%|▌| 8/15 [00:06<00:03,  1.86it/s, est. speed input: 79.88
cessed prompts:  60%|▌| 9/15 [00:07<00:04,  1.46it/s, est. speed input: 77.33
cessed prompts:  67%|▋| 10/15 [00:07<00:02,  1.84it/s, est. speed input: 83.1
cessed prompts:  73%|▋| 11/15 [00:08<00:02,  1.58it/s, est. s

Failed to convert Modern city skyline at night due to not well-formed (invalid token): line 39, column 59, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:03<00:46,  3.30s/it, est. speed input: 18.49
cessed prompts:  13%|▏| 2/15 [00:04<00:23,  1.84s/it, est. speed input: 29.89
cessed prompts:  20%|▏| 3/15 [00:05<00:17,  1.47s/it, est. speed input: 35.93
cessed prompts:  33%|▎| 5/15 [00:05<00:07,  1.39it/s, est. speed input: 57.00
cessed prompts:  40%|▍| 6/15 [00:05<00:05,  1.59it/s, est. speed input: 63.78
cessed prompts:  53%|▌| 8/15 [00:06<00:03,  2.22it/s, est. speed input: 79.25
cessed prompts:  60%|▌| 9/15 [00:07<00:03,  1.65it/s, est. speed input: 76.15
cessed prompts:  67%|▋| 10/15 [00:10<00:06,  1.25s/it, est. speed input: 59.5
cessed prompts:  73%|▋| 11/15 [00:11<00:04,  1.20s/it, est. speed input: 59.4
Processed prompts: 100%|█| 15/15 [00:13<00:00,  1.13it/s, est. speed input: 70.6
Batch prediction:  40%|██████████▊                | 2/5 [00:26<00:39, 13.21s/it]

Failed to convert Rainy day in a small town due to not well-formed (invalid token): line 1, column 2543, Returning default SVG.
Failed to convert Autumn forest with falling leaves due to not well-formed (invalid token): line 42, column 35, Returning default SVG.
Failed to convert Night sky with shooting stars due to not well-formed (invalid token): line 37, column 20, Returning default SVG.
Failed to convert Ocean waves crashing on shore due to not well-formed (invalid token): line 29, column 34, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:03<00:53,  3.80s/it, est. speed input: 16.84
cessed prompts:  13%|▏| 2/15 [00:04<00:22,  1.76s/it, est. speed input: 30.97
cessed prompts:  27%|▎| 4/15 [00:06<00:15,  1.37s/it, est. speed input: 39.40
cessed prompts:  40%|▍| 6/15 [00:06<00:07,  1.17it/s, est. speed input: 54.32
cessed prompts:  47%|▍| 7/15 [00:07<00:06,  1.22it/s, est. speed input: 57.89
cessed prompts:  53%|▌| 8/15 [00:08<00:05,  1.17it/s, est. speed input: 58.83
cessed prompts:  60%|▌| 9/15 [00:08<00:04,  1.42it/s, est. speed input: 64.15
cessed prompts:  67%|▋| 10/15 [00:09<00:02,  1.86it/s, est. speed input: 70.7
cessed prompts:  73%|▋| 11/15 [00:09<00:02,  1.58it/s, est. speed input: 70.7
cessed prompts:  87%|▊| 13/15 [00:12<00:01,  1.07it/s, est. speed input: 66.0
Processed prompts: 100%|█| 15/15 [00:13<00:00,  1.12it/s, est. speed input: 71.5
Batch prediction:  60%|████████████████▏          | 3/5 [00:

Failed to convert Kitchen with fruit bowl on wooden table. due to not well-formed (invalid token): line 1, column 2632, Returning default SVG.
Failed to convert Night sky with stars and crescent moon due to not well-formed (invalid token): line 1, column 2556, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:02<00:38,  2.78s/it, est. speed input: 24.85
cessed prompts:  13%|▏| 2/15 [00:02<00:15,  1.21s/it, est. speed input: 44.70
cessed prompts:  27%|▎| 4/15 [00:04<00:09,  1.11it/s, est. speed input: 60.32
cessed prompts:  33%|▎| 5/15 [00:04<00:07,  1.28it/s, est. speed input: 66.70
cessed prompts:  40%|▍| 6/15 [00:05<00:06,  1.33it/s, est. speed input: 69.12
cessed prompts:  47%|▍| 7/15 [00:05<00:04,  1.77it/s, est. speed input: 78.56
cessed prompts:  53%|▌| 8/15 [00:05<00:03,  2.18it/s, est. speed input: 85.90
cessed prompts:  60%|▌| 9/15 [00:06<00:02,  2.77it/s, est. speed input: 93.95
cessed prompts:  67%|▋| 10/15 [00:07<00:02,  1.67it/s, est. speed input: 88.4
cessed prompts:  73%|▋| 11/15 [00:09<00:04,  1.21s/it, est. speed input: 71.5
cessed prompts:  80%|▊| 12/15 [00:10<00:03,  1.02s/it, est. speed input: 73.8
cessed prompts:  87%|▊| 13/15 [00:10<00:01,  1.25it/s, est. spe

Failed to convert Rainy city street due to not well-formed (invalid token): line 1, column 2418, Returning default SVG.



cessed prompts:   0%| | 0/11 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   9%| | 1/11 [00:03<00:39,  3.91s/it, est. speed input: 16.12
cessed prompts:  27%|▎| 3/11 [00:04<00:10,  1.30s/it, est. speed input: 40.34
cessed prompts:  45%|▍| 5/11 [00:04<00:04,  1.44it/s, est. speed input: 63.60
cessed prompts:  55%|▌| 6/11 [00:05<00:03,  1.55it/s, est. speed input: 69.98
cessed prompts:  64%|▋| 7/11 [00:05<00:02,  1.82it/s, est. speed input: 77.29
cessed prompts:  73%|▋| 8/11 [00:06<00:02,  1.47it/s, est. speed input: 74.79
cessed prompts:  82%|▊| 9/11 [00:08<00:01,  1.08it/s, est. speed input: 68.55
Processed prompts: 100%|█| 11/11 [00:12<00:00,  1.17s/it, est. speed input: 53.7
Batch prediction: 100%|███████████████████████████| 5/5 [01:05<00:00, 13.18s/it]

Failed to convert Starry night sky over mountains due to not well-formed (invalid token): line 42, column 21, Returning default SVG.
Failed to convert Cozy fireplace in winter cabin due to not well-formed (invalid token): line 1, column 2410, Returning default SVG.


In [9]:
df['svg_3']=results

/tmp/ipykernel_10939/1893269863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['svg_3']=results


In [10]:
model.close_model()

In [11]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|███████████████████████████████████████████| 71/71 [00:04<00:00, 15.52it/s]
/tmp/ipykernel_10939/1861074108.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 71/71 [00:07<00:00,  9.07it/s]
/tmp/ipykernel_10939/2425145936.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)


In [14]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

/tmp/ipykernel_10939/1028116435.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3


In [15]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.5680465165900478 mean_aes_score: 0.4523514525991091 combined_score: 0.529481495259735


In [16]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 10
default_svg_score_mean: 1.8209049444384283e-07 default_aes_score_mean: 0.43699469566345217 combined_score: 0.145665019948147


In [17]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 61
non-default_svg_score_mean: 0.6611688665080073 non-default_aes_score_mean: 0.454868953736102 combined_score: 0.5924022289173724
